In [0]:
-- FACT DESCRIBE
DESCRIBE TABLE EXTENDED
  nyc_taxi.gold.fact_yellow_taxi_trip
AS JSON;

-- TABLE METADATA
SELECT
  table_name,
  table_type,
  table_owner,
  comment
FROM nyc_taxi.information_schema.tables
WHERE table_schema = 'gold'
  AND table_name IN (
    'fact_yellow_taxi_trip',
    'dim_date',
    'dim_time',
    'dim_pickup_zone',
    'dim_dropoff_zone',
    'dim_payment_type',
    'dim_rate_code',
    'dim_vendor'
  )
ORDER BY
  CASE table_name
    WHEN 'fact_yellow_taxi_trip' THEN 1
    WHEN 'dim_date' THEN 2
    WHEN 'dim_time' THEN 3
    WHEN 'dim_pickup_zone' THEN 4
    WHEN 'dim_dropoff_zone' THEN 5
    WHEN 'dim_payment_type' THEN 6
    WHEN 'dim_rate_code' THEN 7
    WHEN 'dim_vendor' THEN 8
  END;

-- COLUMN LEVEL METADATA
SELECT
  table_name,
  ordinal_position,
  column_name,
  full_data_type,
  is_nullable,
  comment
FROM nyc_taxi.information_schema.columns
WHERE table_schema = 'gold'
  AND table_name IN (
    'fact_yellow_taxi_trip',
    'dim_date',
    'dim_time',
    'dim_pickup_zone',
    'dim_dropoff_zone',
    'dim_payment_type',
    'dim_rate_code',
    'dim_vendor'
  )
ORDER BY table_name, ordinal_position;

-- FACT SEMANTIC PROFILE
SELECT
  COUNT(*) AS fact_row_count,
  COUNT(DISTINCT trip_key) AS distinct_trip_count,

  MIN(pickup_datetime) AS min_pickup_datetime,
  MAX(pickup_datetime) AS max_pickup_datetime,
  MIN(dropoff_datetime) AS min_dropoff_datetime,
  MAX(dropoff_datetime) AS max_dropoff_datetime,

  COUNT_IF(passenger_count IS NULL)
    AS missing_passenger_count,

  COUNT_IF(trip_distance_miles <= 0)
    AS non_positive_distance,

  COUNT_IF(trip_duration_minutes <= 0)
    AS non_positive_duration,

  COUNT_IF(NOT is_efficiency_kpi_eligible)
    AS efficiency_ineligible_trips,

  COUNT_IF(total_amount < 0)
    AS negative_total_amount_trips,

  COUNT_IF(fare_amount < 0)
    AS negative_fare_amount_trips,

  COUNT_IF(tip_amount < 0)
    AS negative_tip_amount_trips,

  COUNT_IF(is_airport_trip)
    AS airport_trips,

  COUNT_IF(has_reported_electronic_tip)
    AS trips_with_reported_electronic_tip,

  COUNT_IF(
    pickup_date_key = 0
    OR dropoff_date_key = 0
    OR pickup_zone_key = 0
    OR dropoff_zone_key = 0
  ) AS unknown_critical_dimension_trips,

  COUNT_IF(
    payment_type_key = -1
    OR rate_code_key = -1
    OR vendor_key = -1
  ) AS unknown_business_dimension_trips,

  APPROX_COUNT_DISTINCT(route_key)
    AS approximate_distinct_routes

FROM nyc_taxi.gold.fact_yellow_taxi_trip;

-- DIMENSION PROFILES

-- DIM DATE
SELECT
  MIN(full_date) AS min_date,
  MAX(full_date) AS max_date,
  COUNT(*) AS date_member_count,
  COUNT_IF(is_weekend) AS weekend_days,
  COUNT_IF(is_federal_holiday) AS federal_holidays,
  COUNT_IF(is_business_day) AS business_days
FROM nyc_taxi.gold.dim_date
WHERE date_key <> 0;

-- HOLIDAYS
SELECT
  full_date,
  holiday_name,
  holiday_scope
FROM nyc_taxi.gold.dim_date
WHERE is_federal_holiday
ORDER BY full_date;

-- HOURLY
SELECT
  time_key,
  hour_number,
  hour_range,
  day_period,
  business_time_band,
  is_peak_hour
FROM nyc_taxi.gold.dim_time
ORDER BY time_key;

-- PAYMENT
SELECT *
FROM nyc_taxi.gold.dim_payment_type
ORDER BY payment_type_key;

-- FARE
SELECT *
FROM nyc_taxi.gold.dim_rate_code
ORDER BY rate_code_key;

-- VENDOR
SELECT *
FROM nyc_taxi.gold.dim_vendor
ORDER BY vendor_key;

-- GEOGRAPHY AND AIRPORTS
SELECT
  pickup_borough,
  pickup_service_zone,
  COUNT(*) AS zone_count
FROM nyc_taxi.gold.dim_pickup_zone
WHERE pickup_zone_key <> 0
GROUP BY pickup_borough, pickup_service_zone
ORDER BY pickup_borough, pickup_service_zone;

SELECT
  pickup_location_id,
  pickup_borough,
  pickup_zone_name,
  pickup_zone_display_name,
  airport_code
FROM nyc_taxi.gold.dim_pickup_zone
WHERE is_airport_zone
ORDER BY airport_code;

-- DOMAIN DISTRIBUTION
SELECT
  payment.payment_type_name,
  payment.payment_category,
  payment.is_electronic_payment,
  SUM(fact.trip_count) AS trip_count,
  SUM(fact.total_amount) AS total_amount,
  SUM(fact.tip_amount) AS raw_tip_amount
FROM nyc_taxi.gold.fact_yellow_taxi_trip AS fact
JOIN nyc_taxi.gold.dim_payment_type AS payment
  ON fact.payment_type_key = payment.payment_type_key
GROUP BY
  payment.payment_type_name,
  payment.payment_category,
  payment.is_electronic_payment
ORDER BY trip_count DESC;

-- RATE CODE
SELECT
  rate.rate_code_name,
  rate.rate_category,
  rate.airport_code,
  SUM(fact.trip_count) AS trip_count,
  SUM(fact.total_amount) AS total_amount
FROM nyc_taxi.gold.fact_yellow_taxi_trip AS fact
JOIN nyc_taxi.gold.dim_rate_code AS rate
  ON fact.rate_code_key = rate.rate_code_key
GROUP BY
  rate.rate_code_name,
  rate.rate_category,
  rate.airport_code
ORDER BY trip_count DESC;

-- AIRPORTS
SELECT
  COALESCE(airport_trip_code, 'Non-airport') AS airport_trip_segment,
  SUM(trip_count) AS trip_count,
  SUM(total_amount) AS total_amount,
  AVG(total_amount) AS average_total_amount,
  AVG(trip_distance_miles) AS average_distance,
  AVG(trip_duration_minutes) AS average_duration
FROM nyc_taxi.gold.fact_yellow_taxi_trip
GROUP BY COALESCE(airport_trip_code, 'Non-airport')
ORDER BY trip_count DESC;


-- METRIC VIEW BASELINE
SELECT
  SUM(fact.trip_count)
    AS trip_count,

  SUM(fact.total_amount)
    AS total_recorded_amount,

  TRY_DIVIDE(
    SUM(fact.total_amount),
    SUM(fact.trip_count)
  ) AS average_recorded_amount_per_trip,

  SUM(fact.fare_amount)
    AS fare_amount,

  SUM(fact.extra_amount)
    AS extra_amount,

  SUM(fact.mta_tax_amount)
    AS mta_tax_amount,

  SUM(fact.tolls_amount)
    AS tolls_amount,

  SUM(fact.improvement_surcharge_amount)
    AS improvement_surcharge_amount,

  SUM(fact.congestion_surcharge_amount)
    AS congestion_surcharge_amount,

  SUM(fact.cbd_congestion_fee_amount)
    AS cbd_congestion_fee_amount,

  SUM(fact.airport_fee_amount)
    AS airport_fee_amount,

  SUM(fact.tip_amount)
    FILTER (WHERE payment.payment_type_code = 1)
    AS reported_card_tip_amount,

  SUM(fact.trip_count)
    FILTER (WHERE fact.is_efficiency_kpi_eligible)
    AS efficiency_eligible_trip_count,

  TRY_DIVIDE(
    SUM(fact.trip_distance_miles)
      FILTER (WHERE fact.is_efficiency_kpi_eligible),
    SUM(fact.trip_count)
      FILTER (WHERE fact.is_efficiency_kpi_eligible)
  ) AS average_eligible_distance_miles,

  TRY_DIVIDE(
    SUM(fact.trip_duration_minutes)
      FILTER (WHERE fact.is_efficiency_kpi_eligible),
    SUM(fact.trip_count)
      FILTER (WHERE fact.is_efficiency_kpi_eligible)
  ) AS average_eligible_duration_minutes,

  TRY_DIVIDE(
    SUM(fact.trip_distance_miles)
      FILTER (WHERE fact.is_efficiency_kpi_eligible),
    (
      SUM(fact.trip_duration_minutes)
        FILTER (WHERE fact.is_efficiency_kpi_eligible)
    ) / 60.0
  ) AS estimated_average_speed_mph,

  TRY_DIVIDE(
    SUM(fact.total_amount)
      FILTER (WHERE fact.is_efficiency_kpi_eligible),
    SUM(fact.trip_distance_miles)
      FILTER (WHERE fact.is_efficiency_kpi_eligible)
  ) AS recorded_amount_per_mile,

  SUM(CAST(fact.passenger_count AS DOUBLE))
    FILTER (WHERE fact.passenger_count IS NOT NULL)
    AS reported_passenger_count,

  COUNT_IF(fact.passenger_count IS NOT NULL)
    AS trips_with_reported_passenger_count,

  TRY_DIVIDE(
    SUM(CAST(fact.passenger_count AS DOUBLE))
      FILTER (WHERE fact.passenger_count IS NOT NULL),
    COUNT_IF(fact.passenger_count IS NOT NULL)
  ) AS average_reported_passengers,

  TRY_DIVIDE(
    SUM(fact.trip_count)
      FILTER (WHERE payment.payment_type_code = 1),
    SUM(fact.trip_count)
  ) AS credit_card_trip_share,

  TRY_DIVIDE(
    SUM(fact.trip_count)
      FILTER (WHERE fact.is_airport_trip),
    SUM(fact.trip_count)
  ) AS airport_trip_share

FROM nyc_taxi.gold.fact_yellow_taxi_trip AS fact
JOIN nyc_taxi.gold.dim_payment_type AS payment
  ON fact.payment_type_key = payment.payment_type_key;